In [0]:
# =============================================================================
# CARICAMENTO E UNIONE DI TUTTI I CONTINENTI DA DELTA
# =============================================================================

from pyspark.sql import functions as F

# 1. Definisci i percorsi Delta per ciascun continente (adatta se necessario)
continent_paths = {
    "eu": "/Volumes/workspace/weather/continents/eu/",
    "as": "/Volumes/workspace/weather/continents/as/",
    "af": "/Volumes/workspace/weather/continents/af/",
    "oc": "/Volumes/workspace/weather/continents/oc/",
    "na": "/Volumes/workspace/weather/continents/na/",
    "sa": "/Volumes/workspace/weather/continents/sa/"
}

# 2. Carica ciascun Delta e aggiungi colonna "continent" per tracciamento
dfs = {}
for cont_code, path in continent_paths.items():
    try:
        df_temp = spark.read.format("delta").load(path)
        df_temp = df_temp.withColumn("continent", F.lit(cont_code.upper()))  # es. "EU", "AS"...
        dfs[cont_code] = df_temp
        print(f"✓ Caricato {cont_code.upper()}: {df_temp.count()} righe")
    except Exception as e:
        print(f"✗ Errore caricamento {cont_code.upper()}: {str(e)[:100]}...")

# 3. Unione di tutti i DataFrame validi
if dfs:
    df_continents = None
    for cont_code, df in dfs.items():
        if df_continents is None:
            df_continents = df
        else:
            df_continents = df_continents.unionByName(df, allowMissingColumns=True)
    
    print(f"\nUnione completata — righe totali: {df_continents.count()}") 
    display(df_continents.limit(10))
    
    # 4. Salva l'unito in un nuovo Delta
    global_delta_path = "/Volumes/workspace/weather/continents/data/"  # ← CAMBIA SE VUOI UN ALTRO NOM
    df_continents.write.format("delta").mode("overwrite").save(global_delta_path)
    print(f"Salvato unito in Delta: {global_delta_path}")
    
else:
    print("Nessun continente caricato con successo!")

# Opzionale: Verifica colonne e anni
print("\nColonne nel DataFrame unito:", df_continents.columns)
display(df_continents.groupBy("continent", "year").count().orderBy("continent", "year"))

✓ Caricato EU: 108377 righe
✓ Caricato AS: 117590 righe
✓ Caricato AF: 97455 righe
✓ Caricato OC: 61776 righe
✓ Caricato NA: 65504 righe
✓ Caricato SA: 81096 righe

Unione completata — righe totali: 531798


STATION,DATE,LATITUDE,LONGITUDE,ELEVATION,NAME,TEMP,TEMP_ATTRIBUTES,DEWP,DEWP_ATTRIBUTES,SLP,SLP_ATTRIBUTES,STP,STP_ATTRIBUTES,VISIB,VISIB_ATTRIBUTES,WDSP,WDSP_ATTRIBUTES,MXSPD,GUST,MAX,MAX_ATTRIBUTES,MIN,MIN_ATTRIBUTES,PRCP,PRCP_ATTRIBUTES,SNDP,FRSHTT,USAF,year,continent,season,TEMP_c,MAX_c,MIN_c
16020099999,2000-01-01,46.460194,11.326383,240.48,"BOLZANO, IT",23.8,14.0,16.1,14.0,9999.9,0.0,999.9,0.0,6.8,14.0,0.0,7.0,1.0,999.9,35.1,,13.8,,0.0,C,999.9,0,160200,2000,EU,Inverno,-4.555555555555555,1.722222222222223,-10.11111111111111
16020099999,2000-01-02,46.460194,11.326383,240.48,"BOLZANO, IT",24.5,16.0,17.4,16.0,9999.9,0.0,999.9,0.0,6.3,16.0,0.9,9.0,4.1,999.9,36.3,,11.8,,0.0,C,999.9,0,160200,2000,EU,Inverno,-4.166666666666667,2.3888888888888875,-11.222222222222221
16020099999,2000-01-03,46.460194,11.326383,240.48,"BOLZANO, IT",29.1,17.0,20.2,16.0,9999.9,0.0,999.9,0.0,6.4,17.0,0.6,8.0,4.1,999.9,42.1,,16.7,,0.0,C,999.9,0,160200,2000,EU,Inverno,-1.6111111111111103,5.611111111111112,-8.5
16020099999,2000-01-04,46.460194,11.326383,240.48,"BOLZANO, IT",28.5,17.0,19.1,17.0,9999.9,0.0,999.9,0.0,6.2,17.0,0.0,7.0,1.0,999.9,40.6,,17.6,*,0.0,C,999.9,0,160200,2000,EU,Inverno,-1.9444444444444444,4.777777777777779,-8.0
16020099999,2000-01-05,46.460194,11.326383,240.48,"BOLZANO, IT",30.4,16.0,21.9,16.0,9999.9,0.0,999.9,0.0,6.2,16.0,0.0,7.0,1.9,999.9,43.3,,18.0,,0.0,C,999.9,0,160200,2000,EU,Inverno,-0.8888888888888897,6.277777777777776,-7.777777777777778
16020099999,2000-01-06,46.460194,11.326383,240.48,"BOLZANO, IT",30.8,17.0,23.6,17.0,9999.9,0.0,999.9,0.0,6.9,17.0,0.0,6.0,2.9,999.9,42.8,*,17.6,,0.0,B,999.9,0,160200,2000,EU,Inverno,-0.6666666666666663,5.999999999999998,-8.0
16020099999,2000-01-07,46.460194,11.326383,240.48,"BOLZANO, IT",29.7,17.0,21.6,17.0,9999.9,0.0,999.9,0.0,6.2,17.0,0.0,7.0,1.0,999.9,41.0,*,18.9,,0.0,B,999.9,0,160200,2000,EU,Inverno,-1.2777777777777781,5.0,-7.277777777777778
16020099999,2000-01-08,46.460194,11.326383,240.48,"BOLZANO, IT",29.8,15.0,20.4,15.0,9999.9,0.0,999.9,0.0,6.4,15.0,0.0,9.0,2.9,999.9,41.0,,17.8,,0.0,C,999.9,0,160200,2000,EU,Inverno,-1.2222222222222219,5.0,-7.888888888888889
16020099999,2000-01-09,46.460194,11.326383,240.48,"BOLZANO, IT",29.8,15.0,21.6,15.0,9999.9,0.0,999.9,0.0,6.7,15.0,0.0,9.0,1.0,999.9,36.9,,24.8,*,0.0,B,999.9,0,160200,2000,EU,Inverno,-1.2222222222222219,2.7222222222222214,-4.0
16020099999,2000-01-10,46.460194,11.326383,240.48,"BOLZANO, IT",30.9,17.0,23.3,17.0,9999.9,0.0,999.9,0.0,6.4,17.0,0.0,8.0,1.9,999.9,41.0,*,19.6,,0.0,B,999.9,0,160200,2000,EU,Inverno,-0.6111111111111119,5.0,-6.888888888888888


Salvato unito in Delta: /Volumes/workspace/weather/continents/data/

Colonne nel DataFrame unito: ['STATION', 'DATE', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'TEMP', 'TEMP_ATTRIBUTES', 'DEWP', 'DEWP_ATTRIBUTES', 'SLP', 'SLP_ATTRIBUTES', 'STP', 'STP_ATTRIBUTES', 'VISIB', 'VISIB_ATTRIBUTES', 'WDSP', 'WDSP_ATTRIBUTES', 'MXSPD', 'GUST', 'MAX', 'MAX_ATTRIBUTES', 'MIN', 'MIN_ATTRIBUTES', 'PRCP', 'PRCP_ATTRIBUTES', 'SNDP', 'FRSHTT', 'USAF', 'year', 'continent', 'season', 'TEMP_c', 'MAX_c', 'MIN_c']


continent,year,count
AF,2000,3648
AF,2001,3839
AF,2002,3865
AF,2003,3842
AF,2004,4011
AF,2005,3635
AF,2006,3650
AF,2007,3357
AF,2008,3525
AF,2009,3728


In [0]:
# =============================================================================
# Modifica Temp_C corretta e salvataggio 
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Assumi df_all sia già caricato dall'unione precedente
# Se non, ricaricalo
delta_global = "/Volumes/workspace/weather/continents/data/"  # adatta
df_all = spark.read.format("delta").load(delta_global)

# 1. Fix problemi temp_c: filtra valori anomali/mancanti (9999.9 o estremi irreali)
#    Assumi siano in °F se >50, ma filtra prima
df_fixed = df_all.filter(
    (F.col("TEMP") != 9999.9) & (F.col("MAX") != 9999.9) & (F.col("MIN") != 9999.9) &
    (F.col("TEMP") < 150) & (F.col("TEMP") > -100)  # range realistico in °F per sicurezza
)

# Controllo e conversione (se necessario)
temp_stats = df_fixed.agg(F.avg("TEMP").alias("avg")).collect()[0]["avg"]
if temp_stats > 40:
    print("→ Temperature in °F → conversione in °C con filtro estremi")
    df_fixed = df_fixed.withColumn("TEMP_c", (F.col("TEMP") - 32) * 5/9) \
                       .withColumn("MAX_c", (F.col("MAX") - 32) * 5/9) \
                       .withColumn("MIN_c", (F.col("MIN") - 32) * 5/9) \
                       .filter(F.col("TEMP_c") < 60)  # elimina irreali >60°C
else:
    print("→ Temperature già in °C")
    df_fixed = df_fixed.withColumnRenamed("TEMP", "TEMP_c") \
                       .withColumnRenamed("MAX", "MAX_c") \
                       .withColumnRenamed("MIN", "MIN_c")

print("Statistiche TEMP_c fixate:")
display(df_fixed.select("TEMP_c", "MAX_c", "MIN_c").summary())

# 4. Salva l'unito in un nuovo Delta
global_delta_path = "/Volumes/workspace/weather/data/continents1_06_03/"  # ← CAMBIA SE VUOI UN ALTRO NOM
df_fixed.write.format("delta").mode("overwrite").save(global_delta_path)
print(f"Salvato unito in Delta: {global_delta_path}")

# 2. Aggiungi NAME per chiarezza (se non c'è già, assumi da ETL)
df_fixed = df_fixed.withColumn("NAME", F.coalesce(F.col("NAME"), F.col("STATION")))  # fallback su STATION se NAME null
df=df_fixed

→ Temperature in °F → conversione in °C con filtro estremi
Statistiche TEMP_c fixate:


summary,TEMP_c,MAX_c,MIN_c
count,531694,531694,531694
mean,17.805086823122547,22.92877450814354,13.051183471027366
stddev,9.13418918660585,9.738393504232674,9.188627610188863
min,-42.833333333333336,-39.0,-45.77777777777778
25%,12.222222222222221,17.0,7.11111111111111
50%,18.22222222222222,23.77777777777778,13.38888888888889
75%,25.33333333333333,30.77777777777778,20.500000000000004
max,42.166666666666664,51.0,40.0


Salvato unito in Delta: /Volumes/workspace/weather/data/continents1_06_03/


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
# Import 
delta_global = "/Volumes/workspace/weather/data/continents1_06_03/"  
df_fixed= spark.read.format("delta").load(delta_global)
df=df_fixed

In [0]:
# =============================================================================
# CREA COLONNA 'country' (nome esteso) da NAME e mappatura
# =============================================================================

# 1. Mappa ISO2 → Nome paese esteso (dal tuo dizionario)
country_map = {
    # Europa
    "IT": "Italia",
    "FR": "Francia",
    "GM": "Germania",
    "SP": "Spagna",
    "UK": "Regno Unito",
    # Asia
    "CH": "Cina",
    "IN": "India",
    "JA": "Giappone",
    "RS": "Russia",
    "ID": "Indonesia",
    "EN": "Estonia",
    # Africa
    "SF": "Sudafrica",
    "NI": "Nigeria",
    "EG": "Egitto",
    "TZ": "Tanzania",
    "KE" : "Kenya",
    "MO": "Marocco",
    # Nord America
    "US": "Stati Uniti",
    "ES" : "El Salvador",
    "CA": "Canada",
    "MX": "Messico",
    "CU": "Cuba",
    "JM": "Giamaica",
    "GT" : "Guatemala",
    # Sud America
    "BR": "Brasile",
    "AR": "Argentina",
    "CI": "Cile",
    "CO": "Colombia",
    "PE": "Perù",
    # Oceania
    "AS": "Australia",
    "NZ": "Nuova Zelanda",
    "PG": "Papua Nuova Guinea"  # se presente
}


# 2. Crea DataFrame di mapping con nomi NON ambigui
mapping_df = spark.createDataFrame(
    [(code, name) for code, name in country_map.items()],
    ["iso_code_map", "nome_paese_esteso"]
)

# 3. Estrazione sicura del codice ISO2 da NAME
df_with_iso = df.withColumn(
    "part_after_comma",
    F.trim(F.split(F.col("NAME"), ",")[1])  # prende tutto dopo virgola e trimma spazi
).withColumn(
    "iso_code_extracted",
    F.substring(F.col("part_after_comma"), 1, 2)  # primi 2 caratteri
).withColumn(
    "iso_code_extracted",
    F.when(
        (F.length(F.col("iso_code_extracted")) == 2) & 
        (F.col("iso_code_extracted").rlike("^[A-Z]{2}$")),  # solo lettere maiuscole
        F.col("iso_code_extracted")
    ).otherwise(None)
).drop("part_after_comma")  # puliamo

# 4. Join con alias molto chiaro per evitare AMBIGUOUS_REFERENCE
df_with_country = df_with_iso.join(
    mapping_df.alias("map_ref"),
    df_with_iso.iso_code_extracted == F.col("map_ref.iso_code_map"),
    "left_outer"
).withColumn(
    "country",
    F.coalesce(
        F.col("map_ref.nome_paese_esteso"),
        F.col("iso_code_extracted"),
        F.lit("Sconosciuto")
    )
)

# 5. Pulizia finale: drop tutte le colonne temporanee/mappatura
df_with_country = df_with_country.drop(
    "map_ref.iso_code_map",
    "map_ref.nome_paese_esteso",
    "iso_code_extracted"
)

# =============================================================================
# VERIFICA VISIVA (esegui e guarda l'output!)
# =============================================================================

print("Esempi di estrazione country (primi 20 distinti):")
display(df_with_country.select("NAME", "country", "continent").distinct().limit(20))

print("Conteggio per country (ordinato per frequenza):")
display(df_with_country.groupBy("country", "continent").count().orderBy(F.desc("count")))

print("Se ci sono ancora 'Sconosciuto' (primi 10 esempi problematici):")
display(df_with_country.filter(F.col("country") == "Sconosciuto").select("NAME").distinct().limit(10))

Esempi di estrazione country (primi 20 distinti):


NAME,country,continent
"CIAMPINO, IT",Italia,EU
"HAMBURG, GM",Germania,EU
"TEMPELHOF, GM",Germania,EU
"MUNICH AIRPORT, GM",Germania,EU
"TOKYO, JA",Giappone,AS
"CHENNAI INTERNATIONAL, IN",India,AS
"BAIYUN INTERNATIONAL, CH",Cina,AS
"ORLY, FR",Francia,EU
"DYCE, UK",Regno Unito,EU
"BEIJING CAPITAL INTERNATIONAL AIRPORT, CH",Cina,AS


Conteggio per country (ordinato per frequenza):


country,continent,count
Australia,OC,45413
Regno Unito,EU,27357
Cina,AS,27355
India,AS,27304
Italia,EU,26969
Francia,EU,26960
Canada,NA,26273
Indonesia,AS,25875
Sudafrica,AF,25056
Marocco,AF,24787


Se ci sono ancora 'Sconosciuto' (primi 10 esempi problematici):


NAME


In [0]:
# 4. Salva l'unito in un nuovo Delta
global_delta_path = "/Volumes/workspace/weather/data/continents2_06_03/"  # ← CAMBIA SE VUOI UN ALTRO NOM
df_with_country.write.format("delta").mode("overwrite").save(global_delta_path)
print(f"Salvato unito in Delta: {global_delta_path}")